<a href="https://colab.research.google.com/github/OJB-Quantum/Notebooks-for-Ideas/blob/main/Quantum_Circuit_Topology_and_Schematics_with_NetworkX_Graphviz_Schemdraw_Pydot_in_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Authored by Onri Jay Benally (2026)

Open Access (CC-BY-4.0)

In [ ]:
!uv pip install -q networkx pydot schemdraw matplotlib graphviz

In [ ]:
# NetworkX + Graphviz + Schemdraw superconducting parametric amplifiers
#
# Google Colab-ready examples:
#   1. Josephson Parametric Amplifier (JPA)
#   2. Josephson Traveling-Wave Parametric Amplifier (JTWPA)
#   3. Kinetic-Inductance Traveling-Wave Parametric Amplifier (KITWPA)
#
# NetworkX stores circuit topology. Graphviz computes topology placement through
# a direct subprocess interface using safe internal node IDs. Schemdraw renders
# the actual lumped-element circuit symbols. No pydot serialization is used.


# - Imports
import shlex
import shutil
import subprocess
from collections.abc import Callable, Iterable
from pathlib import Path
from typing import Any

import matplotlib as mpl
import matplotlib.pyplot as plt
import networkx as nx
import schemdraw
import schemdraw.elements as elm
from IPython.display import SVG, display

try:
    from google.colab import files
except ImportError:
    files = None

schemdraw.use("matplotlib")
mpl.rcParams["figure.dpi"] = 250

# - Control knobs
SCRIPT_VERSION = "2026-09-02-v3-direct-graphviz"
BASE_OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUTPUT_DIR = BASE_OUTPUT_DIR / "paramp_circuit_diagrams_v3"
GRAPHVIZ_PROGRAM = "neato"
GRAPH_SCALE = 0.045
SHOW_GRAPH_TOPOLOGY = True
DISPLAY_SVG_PREVIEW = True
DISPLAY_PNG_PREVIEW = True
AUTO_DOWNLOAD = False
EXPORT_PNG = True
EXPORT_SVG = True
PNG_DPI = 250
MIN_COMPONENT_LENGTH = 2.0

# Default number of repeated distributed sections.
JTWPA_SECTIONS = 5
KITWPA_SECTIONS = 5

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Running {SCRIPT_VERSION}")


def _safe_graphviz_dot(
    graph: nx.Graph,
) -> tuple[str, dict[str, str]]:
    """Build topology-only DOT with Graphviz-safe internal identifiers.

    The original circuit node names and all component metadata remain entirely
    outside the DOT source. This avoids reserved-word collisions and pydot
    serialization defects.

    Args:
        graph: NetworkX graph containing the electrical topology.

    Returns:
        A DOT source string and a mapping from safe IDs to original node names.
    """
    node_to_safe = {
        str(node): f"gv_{index}"
        for index, node in enumerate(graph.nodes())
    }
    safe_to_node = {
        safe_id: node
        for node, safe_id in node_to_safe.items()
    }

    graph_keyword = "digraph" if graph.is_directed() else "graph"
    edge_operator = "->" if graph.is_directed() else "--"

    lines = [
        f"{graph_keyword} G {{",
        "  graph [overlap=false, splines=true];",
        '  node [shape=point, width=0.05, height=0.05, label=""];',
    ]

    for safe_id in node_to_safe.values():
        lines.append(f"  {safe_id};")

    for start, stop in graph.edges():
        start_id = node_to_safe[str(start)]
        stop_id = node_to_safe[str(stop)]
        lines.append(f"  {start_id} {edge_operator} {stop_id};")

    lines.append("}")
    return "\n".join(lines), safe_to_node


def graphviz_positions(
    graph: nx.Graph,
    prog: str = GRAPHVIZ_PROGRAM,
    scale: float = GRAPH_SCALE,
) -> dict[str, tuple[float, float]]:
    """Compute centered Graphviz coordinates without using pydot.

    Graphviz receives a minimal topology-only DOT graph with safe internal IDs.
    Coordinates are parsed from Graphviz's plain-text output and mapped back to
    the original NetworkX node names.

    Args:
        graph: Electrical topology graph.
        prog: Graphviz layout executable, such as ``neato`` or ``dot``.
        scale: Coordinate scale applied after centering.

    Returns:
        Mapping from original node names to centered 2D coordinates.

    Raises:
        RuntimeError: If Graphviz is unavailable or returns malformed output.
    """
    if graph.number_of_nodes() == 0:
        return {}

    executable = shutil.which(prog)
    if executable is None:
        raise RuntimeError(
            f"Graphviz executable {prog!r} was not found. Run the apt-get "
            "installation cell at the top of this script first."
        )

    dot_source, safe_to_node = _safe_graphviz_dot(graph)
    process = subprocess.run(
        [executable, "-Tplain"],
        input=dot_source,
        text=True,
        capture_output=True,
        check=False,
    )

    if process.returncode != 0:
        raise RuntimeError(
            "Graphviz layout failed.\n"
            f"Executable: {executable}\n"
            f"Return code: {process.returncode}\n"
            f"stderr:\n{process.stderr}\n"
            f"DOT source:\n{dot_source}"
        )

    raw: dict[str, tuple[float, float]] = {}
    for line in process.stdout.splitlines():
        fields = shlex.split(line)
        if len(fields) < 4 or fields[0] != "node":
            continue

        safe_id = fields[1]
        if safe_id not in safe_to_node:
            continue

        original_node = safe_to_node[safe_id]
        raw[original_node] = (float(fields[2]), float(fields[3]))

    expected_nodes = {str(node) for node in graph.nodes()}
    missing_nodes = expected_nodes.difference(raw)
    if missing_nodes:
        raise RuntimeError(
            "Graphviz returned no coordinates for nodes: "
            + ", ".join(sorted(missing_nodes))
        )

    xs = [point[0] for point in raw.values()]
    ys = [point[1] for point in raw.values()]
    x_mid = 0.5 * (min(xs) + max(xs))
    y_mid = 0.5 * (min(ys) + max(ys))

    return {
        node: (
            scale * (x_coord - x_mid),
            scale * (y_coord - y_mid),
        )
        for node, (x_coord, y_coord) in raw.items()
    }


def remap_positions(
    positions: dict[str, tuple[float, float]],
    overrides: dict[str, tuple[float, float]],
) -> dict[str, tuple[float, float]]:
    """Apply circuit-geometry constraints after Graphviz placement."""
    updated = dict(positions)
    updated.update(overrides)
    return updated


def point_distance(
    p1: tuple[float, float],
    p2: tuple[float, float],
) -> float:
    """Return Euclidean distance between two coordinates."""
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    return (dx * dx + dy * dy) ** 0.5


def extend_to_min_length(
    p1: tuple[float, float],
    p2: tuple[float, float],
    minimum_length: float = MIN_COMPONENT_LENGTH,
) -> tuple[float, float]:
    """Extend an endpoint if a symbol span is visually too short."""
    length = point_distance(p1, p2)
    if length == 0.0 or length >= minimum_length:
        return p2

    stretch = minimum_length / length
    return (
        p1[0] + stretch * (p2[0] - p1[0]),
        p1[1] + stretch * (p2[1] - p1[1]),
    )


def add_wire(
    drawing: schemdraw.Drawing,
    p1: tuple[float, float],
    p2: tuple[float, float],
) -> None:
    """Add a wire between arbitrary coordinates."""
    drawing += elm.Line().at(p1).to(p2)


def add_component(
    drawing: schemdraw.Drawing,
    kind: str,
    p1: tuple[float, float],
    p2: tuple[float, float],
    label: str = "",
) -> None:
    """Add a two-terminal circuit element between arbitrary coordinates."""
    end = extend_to_min_length(p1, p2)

    constructors: dict[str, Callable[[], Any]] = {
        "resistor": elm.Resistor,
        "capacitor": elm.Capacitor,
        "inductor": elm.Inductor,
        "josephson": lambda: elm.Josephson(box=True),
        "source_ac": elm.SourceSin,
        "wire": elm.Line,
    }

    if kind not in constructors:
        raise ValueError(f"Unsupported component kind: {kind}")

    element = constructors[kind]().at(p1).to(end)
    if label:
        element = element.label(label)
    drawing += element


def add_junction(
    drawing: schemdraw.Drawing,
    point: tuple[float, float],
    label: str | None = None,
) -> None:
    """Add an electrical junction dot and optional node label."""
    dot = elm.Dot().at(point)
    if label:
        dot = dot.label(label, loc="right")
    drawing += dot


def preview_topology(
    graph: nx.Graph,
    positions: dict[str, tuple[float, float]],
    title: str,
) -> None:
    """Display the NetworkX topology using Graphviz-derived coordinates."""
    if not SHOW_GRAPH_TOPOLOGY:
        return

    plt.figure(figsize=(7.0, 4.8))
    nx.draw_networkx(
        graph,
        pos=positions,
        with_labels=True,
        node_size=950,
        font_size=8,
        width=1.2,
    )
    plt.title(f"{title} | NetworkX + Graphviz topology")
    plt.axis("equal")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def save_and_display(
    drawing: schemdraw.Drawing,
    stem: str,
) -> list[Path]:
    """Export a schematic, display previews, and optionally download it."""
    generated: list[Path] = []

    if EXPORT_SVG:
        svg_path = OUTPUT_DIR / f"{stem}.svg"
        drawing.save(str(svg_path))
        generated.append(svg_path)

        if DISPLAY_SVG_PREVIEW:
            display(SVG(filename=str(svg_path)))

    if EXPORT_PNG:
        png_path = OUTPUT_DIR / f"{stem}.png"
        drawing.save(str(png_path), dpi=PNG_DPI)
        generated.append(png_path)

        if DISPLAY_PNG_PREVIEW:
            image = plt.imread(png_path)
            plt.figure(figsize=(10.0, 5.4))
            plt.imshow(image)
            plt.axis("off")
            plt.tight_layout()
            plt.show()

    if AUTO_DOWNLOAD and files is not None:
        for path in generated:
            files.download(str(path))

    return generated


def make_jpa() -> list[Path]:
    """Create a lumped reflection-mode Josephson parametric amplifier."""
    graph = nx.Graph()
    graph.add_edge("PORT", "COUPLER", kind="capacitor", label="Cc")
    graph.add_edge("COUPLER", "RESONATOR", kind="wire", label="")
    graph.add_edge("RESONATOR", "JJ_GND", kind="josephson", label="JJ")
    graph.add_edge("RESONATOR", "C_GND", kind="capacitor", label="C")
    graph.add_edge("RESONATOR", "PUMP", kind="inductor", label="Lb")
    graph.add_edge("JJ_GND", "GROUND", kind="wire", label="")
    graph.add_edge("C_GND", "GROUND", kind="wire", label="")
    graph.add_edge("PUMP", "PUMP_PORT", kind="wire", label="")

    graphviz_pos = graphviz_positions(graph, prog="neato")
    topology_pos = remap_positions(
        graphviz_pos,
        {
            "PORT": (-6.0, 0.0),
            "COUPLER": (-3.7, 0.0),
            "RESONATOR": (0.0, 0.0),
            "JJ_GND": (0.0, -2.8),
            "C_GND": (3.0, -2.8),
            "GROUND": (1.5, -4.3),
            "PUMP": (0.0, 2.8),
            "PUMP_PORT": (3.5, 2.8),
        },
    )
    preview_topology(graph, topology_pos, "Josephson parametric amplifier")

    drawing = schemdraw.Drawing()
    drawing.config(unit=3.0, fontsize=12)

    port = topology_pos["PORT"]
    coupler = topology_pos["COUPLER"]
    resonator = topology_pos["RESONATOR"]
    jj_ground = topology_pos["JJ_GND"]
    c_ground = topology_pos["C_GND"]
    ground = topology_pos["GROUND"]
    pump = topology_pos["PUMP"]
    pump_port = topology_pos["PUMP_PORT"]

    add_wire(drawing, (-8.2, 0.0), port)
    drawing += elm.Label().at((-8.2, 0.55)).label(
        r"signal / reflected output"
    )
    add_component(drawing, "capacitor", port, coupler, r"$C_c$")
    add_wire(drawing, coupler, resonator)

    add_component(drawing, "josephson", resonator, jj_ground, r"$I_c$")
    add_component(drawing, "capacitor", resonator, c_ground, r"$C$")
    add_wire(drawing, jj_ground, ground)
    add_wire(drawing, c_ground, ground)
    drawing += elm.Ground().at(ground)

    add_component(drawing, "inductor", resonator, pump, r"$L_b$")
    add_wire(drawing, pump, pump_port)
    drawing += elm.Label().at((4.8, 2.8)).label(r"pump / flux-bias port")

    add_junction(drawing, resonator, r"$V_r$")
    drawing += elm.Label().at((0.0, 4.0)).label(
        r"JPA: nonlinear Josephson resonator"
    )

    return save_and_display(drawing, "01_josephson_parametric_amplifier")


def _distributed_positions(
    node_prefix: str,
    ground_prefix: str,
    num_sections: int,
    x0: float = -7.0,
    dx: float = 2.7,
) -> dict[str, tuple[float, float]]:
    """Create constrained ladder coordinates for a traveling-wave circuit."""
    positions: dict[str, tuple[float, float]] = {}
    for index in range(num_sections + 1):
        positions[f"{node_prefix}{index}"] = (x0 + dx * index, 0.0)
    for index in range(1, num_sections + 1):
        positions[f"{ground_prefix}{index}"] = (x0 + dx * index, -2.8)
    return positions


def make_jtwpa(num_sections: int = JTWPA_SECTIONS) -> list[Path]:
    """Create a Josephson traveling-wave parametric amplifier ladder."""
    if num_sections < 1:
        raise ValueError("JTWPA_SECTIONS must be at least 1.")

    graph = nx.Graph()
    nodes = [f"J{index}" for index in range(num_sections + 1)]
    grounds = [f"JG{index}" for index in range(1, num_sections + 1)]

    for index in range(num_sections):
        graph.add_edge(
            nodes[index],
            nodes[index + 1],
            kind="josephson",
            label=f"JJ{index + 1}",
        )
        graph.add_edge(
            nodes[index + 1],
            grounds[index],
            kind="capacitor",
            label=f"C{index + 1}",
        )

    graphviz_pos = graphviz_positions(graph, prog="dot")
    topology_pos = remap_positions(
        graphviz_pos,
        _distributed_positions("J", "JG", num_sections),
    )
    preview_topology(
        graph,
        topology_pos,
        "Josephson traveling-wave parametric amplifier",
    )

    drawing = schemdraw.Drawing()
    drawing.config(unit=2.8, fontsize=11)

    x0 = -7.0
    dx = 2.7
    bus_y = -2.8
    bus_end_x = x0 + dx * num_sections

    add_wire(drawing, (x0 - 2.8, 0.0), topology_pos["J0"])
    drawing += elm.Label().at((x0 - 2.7, 0.65)).label(
        r"signal + pump"
    )

    for index in range(num_sections):
        start = topology_pos[f"J{index}"]
        stop = topology_pos[f"J{index + 1}"]
        shunt = topology_pos[f"JG{index + 1}"]

        add_component(
            drawing,
            "josephson",
            start,
            stop,
            rf"$JJ_{{{index + 1}}}$",
        )
        add_component(
            drawing,
            "capacitor",
            stop,
            shunt,
            rf"$C_{{{index + 1}}}$",
        )
        add_wire(drawing, shunt, (bus_end_x, bus_y))
        add_junction(drawing, stop)

    add_wire(drawing, (x0, bus_y), (bus_end_x, bus_y))
    drawing += elm.Ground().at((bus_end_x, bus_y))
    add_wire(
        drawing,
        topology_pos[f"J{num_sections}"],
        (bus_end_x + 2.8, 0.0),
    )
    drawing += elm.Label().at((bus_end_x + 2.5, 0.65)).label(r"amplified output")
    drawing += elm.Label().at((0.0, 2.4)).label(
        r"JTWPA: distributed Josephson-junction transmission line"
    )

    return save_and_display(
        drawing,
        "02_josephson_traveling_wave_parametric_amplifier",
    )


def make_kitwpa(num_sections: int = KITWPA_SECTIONS) -> list[Path]:
    """Create a kinetic-inductance traveling-wave parametric amplifier ladder."""
    if num_sections < 1:
        raise ValueError("KITWPA_SECTIONS must be at least 1.")

    graph = nx.Graph()
    nodes = [f"K{index}" for index in range(num_sections + 1)]
    grounds = [f"KG{index}" for index in range(1, num_sections + 1)]

    for index in range(num_sections):
        graph.add_edge(
            nodes[index],
            nodes[index + 1],
            kind="inductor",
            label=f"Lk{index + 1}",
        )
        graph.add_edge(
            nodes[index + 1],
            grounds[index],
            kind="capacitor",
            label=f"C{index + 1}",
        )

    graphviz_pos = graphviz_positions(graph, prog="dot")
    topology_pos = remap_positions(
        graphviz_pos,
        _distributed_positions("K", "KG", num_sections),
    )
    preview_topology(
        graph,
        topology_pos,
        "Kinetic-inductance traveling-wave parametric amplifier",
    )

    drawing = schemdraw.Drawing()
    drawing.config(unit=2.8, fontsize=11)

    x0 = -7.0
    dx = 2.7
    bus_y = -2.8
    bus_end_x = x0 + dx * num_sections

    add_wire(drawing, (x0 - 2.8, 0.0), topology_pos["K0"])
    drawing += elm.Label().at((x0 - 2.7, 0.65)).label(
        r"signal + pump"
    )

    for index in range(num_sections):
        start = topology_pos[f"K{index}"]
        stop = topology_pos[f"K{index + 1}"]
        shunt = topology_pos[f"KG{index + 1}"]

        add_component(
            drawing,
            "inductor",
            start,
            stop,
            rf"$L_{{k,{index + 1}}}(I)$",
        )
        add_component(
            drawing,
            "capacitor",
            stop,
            shunt,
            rf"$C_{{{index + 1}}}$",
        )
        add_wire(drawing, shunt, (bus_end_x, bus_y))
        add_junction(drawing, stop)

    add_wire(drawing, (x0, bus_y), (bus_end_x, bus_y))
    drawing += elm.Ground().at((bus_end_x, bus_y))
    add_wire(
        drawing,
        topology_pos[f"K{num_sections}"],
        (bus_end_x + 2.8, 0.0),
    )
    drawing += elm.Label().at((bus_end_x + 2.5, 0.65)).label(r"amplified output")
    drawing += elm.Label().at((0.0, 2.4)).label(
        r"KITWPA: nonlinear kinetic-inductance transmission line"
    )

    return save_and_display(
        drawing,
        "03_kinetic_inductance_traveling_wave_parametric_amplifier",
    )


def validate_graphviz_adapter() -> None:
    """Exercise Graphviz with a reserved-word node and component metadata."""
    test_graph = nx.Graph()
    test_graph.add_edge("NODE", "graph", kind="josephson", label="JJ; test")
    test_graph.add_edge("graph", "edge", kind="capacitor", label="C")
    test_positions = graphviz_positions(test_graph, prog=GRAPHVIZ_PROGRAM)

    if set(test_positions) != {"NODE", "graph", "edge"}:
        raise RuntimeError("Graphviz adapter self-test returned incorrect nodes.")

    print("Graphviz safe-ID adapter self-test: PASS")


def download_all(paths: Iterable[Path]) -> None:
    """Download all generated files from a Google Colab runtime."""
    if files is None:
        print("google.colab.files is unavailable outside Colab.")
        return

    for path in paths:
        files.download(str(path))


validate_graphviz_adapter()

all_paths: list[Path] = []
all_paths.extend(make_jpa())
all_paths.extend(make_jtwpa())
all_paths.extend(make_kitwpa())

print("\nGenerated files:")
for output_path in all_paths:
    print(f"  {output_path}")

# Optional manual download in Colab:
# download_all(all_paths)